In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("CalculIndicateursRetards") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.4.1") \
    .config("spark.cassandra.connection.host", "cassandra") \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .getOrCreate()

df_nettoye = spark.read.parquet("/home/jovyan/data/flights_clean.parquet")

print("Nombre de lignes chargées :", df_nettoye.count())
df_nettoye.printSchema()
df_nettoye.show(5)

Nombre de lignes chargées : 149343
root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- Cancelled: integer (nullable = true)

+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
|Year|Month|DayofMonth|DayOfWeek|UniqueCarrier|Origin|Dest|ArrDelay|DepDelay|Cancelled|
+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
|2008|    1|         3|        4|           WN|   MDW| TPA|     4.0|    20.0|        0|
|2008|    1|         3|        4|           WN|   SMF| PHX|     1.0|     9.0|        0|
|2008|    1|         3|        4|           WN|   TPA| PIT|    19.0|    15.0|        0|
|2008|    1|         6|       

#### taux de retard par compagnie 

In [2]:
indicateur_compagnie = df_nettoye.groupBy("UniqueCarrier").agg(
    F.count("*").alias("nb_vols"),
    F.sum(F.when(F.col("ArrDelay") > 15, 1).otherwise(0)).alias("nb_retards"),
    F.round(F.avg("ArrDelay"), 2).alias("retard_moyen")
).withColumn(
    "taux_retard",
    F.round(F.col("nb_retards") / F.col("nb_vols") * 100, 2)
).orderBy(F.col("taux_retard").desc())

indicateur_compagnie.show(30, truncate=False)

+-------------+-------+----------+------------+-----------+
|UniqueCarrier|nb_vols|nb_retards|retard_moyen|taux_retard|
+-------------+-------+----------+------------+-----------+
|YV           |5111   |3772      |53.9        |73.8       |
|OH           |4096   |3019      |51.26       |73.71      |
|B6           |4268   |2940      |56.33       |68.88      |
|NW           |6039   |4113      |45.47       |68.11      |
|XE           |8021   |5461      |50.2        |68.08      |
|AA           |14765  |10051     |46.45       |68.07      |
|9E           |3966   |2694      |47.47       |67.93      |
|EV           |6381   |4295      |47.32       |67.31      |
|MQ           |10926  |7335      |45.62       |67.13      |
|OO           |10227  |6776      |45.61       |66.26      |
|UA           |10769  |7113      |47.84       |66.05      |
|FL           |5473   |3482      |44.37       |63.62      |
|DL           |8891   |5496      |40.14       |61.82      |
|AS           |2920   |1761      |36.21 

#### taux de retard par aeroport :

In [3]:
indicateur_aeroport = df_nettoye.groupBy("Origin").agg(
    F.count("*").alias("nb_vols"),
    F.sum(F.when(F.col("ArrDelay") > 15, 1).otherwise(0)).alias("nb_retards"),
    F.round(F.avg("ArrDelay"), 2).alias("retard_moyen")
).withColumn(
    "taux_retard",
    F.round(F.col("nb_retards") / F.col("nb_vols") * 100, 2)
).filter(F.col("nb_vols") >= 50) \
 .orderBy(F.col("taux_retard").desc())

indicateur_aeroport.show(20, truncate=False)

+------+-------+----------+------------+-----------+
|Origin|nb_vols|nb_retards|retard_moyen|taux_retard|
+------+-------+----------+------------+-----------+
|ILM   |60     |53        |68.03       |88.33      |
|EGE   |71     |60        |107.86      |84.51      |
|CHA   |82     |67        |78.43       |81.71      |
|MGM   |54     |44        |62.87       |81.48      |
|ABE   |73     |59        |66.92       |80.82      |
|ACV   |95     |76        |58.57       |80.0       |
|JAC   |68     |54        |76.0        |79.41      |
|AZO   |66     |51        |60.05       |77.27      |
|GRB   |118    |91        |62.8        |77.12      |
|SGF   |171    |131       |73.14       |76.61      |
|FAR   |98     |75        |65.64       |76.53      |
|SYR   |237    |181       |57.1        |76.37      |
|DAB   |67     |51        |62.96       |76.12      |
|FWA   |112    |85        |71.59       |75.89      |
|BTV   |142    |107       |58.37       |75.35      |
|FSD   |101    |76        |67.29       |75.25 

#### Taux de retard par jour de la semaine

In [4]:
indicateur_jour = df_nettoye.groupBy("DayOfWeek").agg(
    F.count("*").alias("nb_vols"),
    F.sum(F.when(F.col("ArrDelay") > 15, 1).otherwise(0)).alias("nb_retards"),
    F.round(F.avg("ArrDelay"), 2).alias("retard_moyen")
).withColumn(
    "taux_retard",
    F.round(F.col("nb_retards") / F.col("nb_vols") * 100, 2)
).orderBy("DayOfWeek")

indicateur_jour.show(7, truncate=False)

+---------+-------+----------+------------+-----------+
|DayOfWeek|nb_vols|nb_retards|retard_moyen|taux_retard|
+---------+-------+----------+------------+-----------+
|1        |22259  |13979     |41.76       |62.8       |
|2        |20241  |12909     |43.9        |63.78      |
|3        |20155  |12631     |40.25       |62.67      |
|4        |22229  |14074     |41.38       |63.31      |
|5        |25046  |16166     |43.42       |64.55      |
|6        |17244  |10326     |39.9        |59.88      |
|7        |22169  |14090     |44.75       |63.56      |
+---------+-------+----------+------------+-----------+



In [5]:
from pyspark.sql.types import StringType

noms_jours = {
    1: "Lundi", 2: "Mardi", 3: "Mercredi", 4: "Jeudi",
    5: "Vendredi", 6: "Samedi", 7: "Dimanche"
}

mapping_expr = F.create_map([F.lit(x) for pair in noms_jours.items() for x in pair])

indicateur_jour = df_nettoye.groupBy("DayOfWeek").agg(
    F.count("*").alias("nb_vols"),
    F.sum(F.when(F.col("ArrDelay") > 15, 1).otherwise(0)).alias("nb_retards"),
    F.round(F.avg("ArrDelay"), 2).alias("retard_moyen")
).withColumn(
    "taux_retard",
    F.round(F.col("nb_retards") / F.col("nb_vols") * 100, 2)
).withColumn(
    "nom_jour",
    mapping_expr[F.col("DayOfWeek")]
).orderBy("DayOfWeek")

indicateur_jour.select("DayOfWeek", "nom_jour", "nb_vols", "nb_retards", "retard_moyen", "taux_retard") \
    .show(7, truncate=False)

+---------+--------+-------+----------+------------+-----------+
|DayOfWeek|nom_jour|nb_vols|nb_retards|retard_moyen|taux_retard|
+---------+--------+-------+----------+------------+-----------+
|1        |Lundi   |22259  |13979     |41.76       |62.8       |
|2        |Mardi   |20241  |12909     |43.9        |63.78      |
|3        |Mercredi|20155  |12631     |40.25       |62.67      |
|4        |Jeudi   |22229  |14074     |41.38       |63.31      |
|5        |Vendredi|25046  |16166     |43.42       |64.55      |
|6        |Samedi  |17244  |10326     |39.9        |59.88      |
|7        |Dimanche|22169  |14090     |44.75       |63.56      |
+---------+--------+-------+----------+------------+-----------+



#### l'indicateur croisé aéroport + jour

In [6]:
indicateur_aeroport_jour = df_nettoye.groupBy("Origin", "DayOfWeek").agg(
    F.count("*").alias("nb_vols"),
    F.sum(F.when(F.col("ArrDelay") > 15, 1).otherwise(0)).alias("nb_retards"),
    F.round(F.avg("ArrDelay"), 2).alias("retard_moyen")
).withColumn(
    "taux_retard",
    F.round(F.col("nb_retards") / F.col("nb_vols") * 100, 2)
).withColumn(
    "nom_jour",
    mapping_expr[F.col("DayOfWeek")]
).filter(F.col("nb_vols") >= 10) \
 .orderBy("Origin", "DayOfWeek")

indicateur_aeroport_jour.show(15, truncate=False)

+------+---------+-------+----------+------------+-----------+--------+
|Origin|DayOfWeek|nb_vols|nb_retards|retard_moyen|taux_retard|nom_jour|
+------+---------+-------+----------+------------+-----------+--------+
|ABE   |1        |11     |10        |112.36      |90.91      |Lundi   |
|ABE   |2        |10     |9         |61.7        |90.0       |Mardi   |
|ABE   |3        |13     |10        |57.77       |76.92      |Mercredi|
|ABE   |4        |12     |9         |47.5        |75.0       |Jeudi   |
|ABE   |6        |10     |8         |67.0        |80.0       |Samedi  |
|ABI   |2        |12     |10        |121.58      |83.33      |Mardi   |
|ABQ   |1        |107    |56        |33.39       |52.34      |Lundi   |
|ABQ   |2        |94     |52        |36.16       |55.32      |Mardi   |
|ABQ   |3        |126    |64        |27.5        |50.79      |Mercredi|
|ABQ   |4        |126    |71        |32.29       |56.35      |Jeudi   |
|ABQ   |5        |148    |82        |37.48       |55.41      |Ve

In [7]:
indicateur_compagnie.write.mode("overwrite").parquet("/home/jovyan/data/indicateur_compagnie.parquet")
indicateur_aeroport.write.mode("overwrite").parquet("/home/jovyan/data/indicateur_aeroport.parquet")
indicateur_jour.write.mode("overwrite").parquet("/home/jovyan/data/indicateur_jour.parquet")
indicateur_aeroport_jour.write.mode("overwrite").parquet("/home/jovyan/data/indicateur_aeroport_jour.parquet")

print("Les 4 indicateurs ont été sauvegardés.")

Les 4 indicateurs ont été sauvegardés.
